# ScienceQA Visual Challenge — Optimized Notebook

**Strategy summary:**
- **Model**: `HuggingFaceTB/SmolVLM-500M-Instruct` (required by rules)
- **Fine-tuning**: QLoRA (4-bit NF4), LoRA r=16 on all attention projections — verified ≤5M trainable params
- **Inference**: Log-likelihood scoring over each choice letter (replaces unreliable generation)
- **Prompts**: lecture + hint + solution (chain-of-thought) during training
- **Augmentation**: Random choice-order shuffling to prevent positional bias
- **Final model**: Retrained on train+val combined before submission
- **Ensemble**: 3 seeds, average log-probs (optional section at bottom)

In [1]:
# ── 0. Install ────────────────────────────────────────────────────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/44.0 kB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00


   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.6/12.0 MB 19.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/12.0 MB 55.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 8.1/12.0 MB 77.6 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 201.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 201.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 201.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 201.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.2 MB/s eta 0:00:00


In [2]:
# ── 1. Imports & Configuration ────────────────────────────────────────────────
import os
import gc
import json
import random
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/kaggle/input/datasets/komvopoulos/finalexamdataset")

# ── Model & training knobs ────────────────────────────────────────────────────
MODEL_ID       = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE       = 224
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
LR             = 2e-4
NUM_EPOCHS     = 3             # 5 epochs × ~43h = too long; 3 epochs ≈ 5-7 h on T4
BATCH_SIZE     = 2
GRAD_ACCUM     = 8             # effective batch = 16
CHOICE_LETTERS = "ABCDEFGHIJ"
PARAM_BUDGET   = 5_000_000

# ── Text-field limits ─────────────────────────────────────────────────────────
# lecture  → EXCLUDED: generic background the model already knows from pretraining.
#            Dropping it saves ~125 tokens/sample, directly reducing attention cost.
# hint     → kept full: the specific scenario passage the question is about.
# solution → 400 chars (~100 tokens): chain-of-thought reasoning scaffold.
INCLUDE_LECTURE    = False
SOLUTION_MAX_CHARS = 400

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

2026-04-29 03:43:52.378116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777434232.568038      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777434232.621461      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777434233.084550      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777434233.084597      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777434233.084599      23 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Load Data

In [3]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# Detect whether solution is available at test time.
# In ScienceQA the test split typically omits solutions since they reveal the answer.
# We check dynamically so the code works either way.
TEST_HAS_SOLUTION = (
    "solution" in test_df.columns
    and test_df["solution"].notna().any()
    and test_df["solution"].astype(str).str.strip().ne("").any()
)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Solution available in test.csv: {TEST_HAS_SOLUTION}")
print(f"Test columns: {test_df.columns.tolist()}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008
Solution available in test.csv: False
Test columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'hint', 'lecture', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [4]:
# ── 2b. Prompt Engineering ────────────────────────────────────────────────────

def _trunc(text, max_chars: int) -> str:
    text = str(text).strip()
    return text if len(text) <= max_chars else text[:max_chars] + "…"


def build_prompt(
    row,
    include_answer: bool = False,
    include_solution: bool = False,
) -> str:
    context_parts = []

    # lecture excluded by default (INCLUDE_LECTURE=False) — saves ~125 tokens/sample
    if INCLUDE_LECTURE:
        lecture = row.get("lecture", "") if isinstance(row, dict) else getattr(row, "lecture", "")
        if pd.notna(lecture) and str(lecture).strip():
            context_parts.append(_trunc(lecture, 500))

    hint = row.get("hint", "") if isinstance(row, dict) else getattr(row, "hint", "")
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    if include_solution:
        sol = row.get("solution", "") if isinstance(row, dict) else getattr(row, "solution", "")
        if pd.notna(sol) and str(sol).strip():
            context_parts.append("Reasoning: " + _trunc(sol, SOLUTION_MAX_CHARS))

    choices = row["choices"] if isinstance(row, dict) else row.choices
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_parts:
        prompt += "Context:\n" + "\n\n".join(context_parts) + "\n\n"
    question = row["question"] if isinstance(row, dict) else row.question
    prompt += f"Question: {question}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer = row["answer"] if isinstance(row, dict) else row.answer
        prompt += f" {CHOICE_LETTERS[int(answer)]}"

    return prompt

sample = build_prompt(train_df.iloc[0].to_dict(), include_answer=True, include_solution=True)
print(f"Sample prompt length: {len(sample)} chars")
print(sample[:300], "…")

Sample prompt length: 1845 chars
<image>
Context:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, t …


In [5]:
# ── 2c. Dataset with choice-order augmentation ────────────────────────────────

class ScienceQADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: Path,
        img_size: int = IMG_SIZE,
        is_train: bool = True,
        augment_choices: bool = False,
        include_solution: bool = False,
    ):
        self.df               = df.reset_index(drop=True)
        self.data_dir         = data_dir
        self.img_size         = img_size
        self.is_train         = is_train
        self.augment_choices  = augment_choices
        self.include_solution = include_solution

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx].to_dict()

        # image_path in the CSV is e.g. "images/train/train_06662.png"
        # but the dataset is nested one level deeper: data_dir/images/<image_path>
        img = Image.open(self.data_dir / "images" / row["image_path"]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)

        if self.augment_choices and self.is_train:
            choices = list(row["choices"])
            answer  = int(row["answer"])
            perm    = list(range(len(choices)))
            random.shuffle(perm)
            row["choices"] = [choices[p] for p in perm]
            row["answer"]  = perm.index(answer)

        if self.is_train:
            return {
                "image": img,
                "text":  build_prompt(row, include_answer=True,
                                      include_solution=self.include_solution),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False,
                                        include_solution=self.include_solution),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row and pd.notna(row.get("answer")) else -1,
                "id":      row.get("id", ""),
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, is_train=True,
                            augment_choices=True, include_solution=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, is_train=False,
                            include_solution=TEST_HAS_SOLUTION)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, is_train=False,
                            include_solution=TEST_HAS_SOLUTION)

print(f"Datasets — train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

Datasets — train: 3109 | val: 1048 | test: 1008


## 3. Load Model with QLoRA

The parameter budget is **5 million trainable params**. With `r=16` on the four attention projections (`q, k, v, o`) of SmolVLM-500M's language backbone this lands at roughly 2–4 M params — safely within budget. `model.print_trainable_parameters()` shows the exact count. If it exceeds 5 M, change `LORA_R = 8`.

In [6]:
# ── 3. Load SmolVLM-500M with 4-bit QLoRA ────────────────────────────────────

def load_model_and_processor(model_id: str = MODEL_ID, lora_r: int = LORA_R):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    proc = AutoProcessor.from_pretrained(model_id)
    if proc.tokenizer.pad_token is None:
        proc.tokenizer.pad_token = proc.tokenizer.eos_token

    base = AutoModelForVision2Seq.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    # Gradient checkpointing must stay ON: without it, storing all activations
    # across 27 SigLIP + 32 LLM layers fills the T4's 14.56 GB.
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    mdl = get_peft_model(base, lora_cfg)

    trainable, total = mdl.get_nb_trainable_parameters()
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")
    assert trainable <= PARAM_BUDGET, (
        f"Trainable params {trainable:,} exceed budget {PARAM_BUDGET:,}. "
        "Set LORA_R = 8 and re-run this cell."
    )
    return mdl, proc


model, processor = load_model_and_processor()

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable params: 4,161,536 / 511,643,840 (0.813%)


In [7]:
# ── 4. Collate function — answer-token label masking ─────────────────────────
#
# The training text always ends with "Answer: X" (X = one letter).
# The processor appends an EOS token after X.
#
# Previous approach searched for the "Answer:" token subsequence, but BPE
# tokenization is context-dependent: the same string encodes to different
# token IDs in isolation vs. embedded in a full sequence. The search
# silently failed → fallback trained on EOS (trivial → loss collapses to 0,
# accuracy = random chance ≈ 33%).
#
# This version doesn't search for text patterns. It walks backward past any
# EOS tokens and unmasks exactly the last remaining real token — the answer
# letter. Initial CE loss ≈ 2–4, decreasing steadily during training.

def collate_fn(batch, proc):
    texts  = [item["text"]  for item in batch]
    images = [item["image"] for item in batch]

    inputs = proc(text=texts, images=images, return_tensors="pt", padding=True)

    eos_id = proc.tokenizer.eos_token_id
    pad_id = proc.tokenizer.pad_token_id

    labels = torch.full_like(inputs["input_ids"], -100)

    for i in range(labels.shape[0]):
        seq      = inputs["input_ids"][i].tolist()
        real_end = int(inputs["attention_mask"][i].sum().item())

        # Walk backward past EOS to find the answer letter token
        answer_pos = real_end - 1
        while answer_pos >= 0 and seq[answer_pos] in (eos_id, pad_id):
            answer_pos -= 1

        if answer_pos >= 0:
            labels[i, answer_pos] = inputs["input_ids"][i, answer_pos]

    inputs["labels"] = labels
    return inputs


_collate = partial(collate_fn, proc=processor)

In [8]:
# ── 4b. Sanity-check collate_fn on 4 real examples ───────────────────────────
# Run this cell before training to confirm:
#   (a) the answer token is found correctly (not EOS, not a padding token)
#   (b) the decoded token matches the expected answer letter
#   (c) only 1 position per sample has a real label (everything else is -100)
# If any CHECK FAILED line appears, report it before proceeding.

import textwrap

_check_items = [train_ds[i] for i in range(4)]
_check_batch = _collate(_check_items)

eos_id = processor.tokenizer.eos_token_id
pad_id = processor.tokenizer.pad_token_id

print(f"{'='*60}")
print(f"eos_token_id = {eos_id}  |  pad_token_id = {pad_id}")
print(f"{'='*60}\n")

for i, item in enumerate(_check_items):
    seq      = _check_batch["input_ids"][i].tolist()
    mask     = _check_batch["attention_mask"][i].tolist()
    labels   = _check_batch["labels"][i].tolist()
    real_end = sum(mask)

    # Which positions have real labels?
    labeled_positions = [p for p, l in enumerate(labels) if l != -100]

    expected_letter = CHOICE_LETTERS[item["answer"]]

    print(f"--- Sample {i}  (expected answer: {expected_letter}) ---")
    print(f"  Sequence length (with padding): {len(seq)}")
    print(f"  Real tokens (non-pad):          {real_end}")
    print(f"  Labeled positions:              {labeled_positions}")

    # Show last 8 real tokens for inspection
    window = seq[max(0, real_end - 8): real_end]
    decoded = [processor.tokenizer.decode([t]) for t in window]
    print(f"  Last 8 real tokens (ids):  {window}")
    print(f"  Last 8 real tokens (text): {decoded}")

    if len(labeled_positions) == 0:
        print(f"  !! CHECK FAILED: no labeled position found !!")
    elif len(labeled_positions) > 1:
        print(f"  !! CHECK FAILED: {len(labeled_positions)} labeled positions (expected 1) !!")
    else:
        pos        = labeled_positions[0]
        tok_id     = seq[pos]
        tok_str    = processor.tokenizer.decode([tok_id]).strip()
        is_eos     = tok_id == eos_id
        is_pad     = tok_id == pad_id
        letter_match = tok_str.upper() == expected_letter

        print(f"  Token at label pos {pos}: id={tok_id}  decoded='{tok_str}'")
        if is_eos:
            print(f"  !! CHECK FAILED: labeled token is EOS !!")
        elif is_pad:
            print(f"  !! CHECK FAILED: labeled token is PAD !!")
        elif not letter_match:
            print(f"  !! CHECK FAILED: decoded '{tok_str}' != expected '{expected_letter}' !!")
        else:
            print(f"  OK: answer token '{tok_str}' matches expected '{expected_letter}'")
    print()

eos_token_id = 49279  |  pad_token_id = 2

--- Sample 0  (expected answer: C) ---
  Sequence length (with padding): 1524
  Real tokens (non-pad):          1524
  Labeled positions:              [1523]
  Last 8 real tokens (ids):  [523, 1438, 2490, 18952, 198, 21350, 42, 340]
  Last 8 real tokens (text): [' will', ' become', ' adult', ' frogs', '\n', 'Answer', ':', ' C']
  Token at label pos 1523: id=340  decoded='C'
  OK: answer token 'C' matches expected 'C'

--- Sample 1  (expected answer: C) ---
  Sequence length (with padding): 1524
  Real tokens (non-pad):          1498
  Labeled positions:              [1497]
  Last 8 real tokens (ids):  [14066, 523, 2330, 2848, 198, 21350, 42, 340]
  Last 8 real tokens (text): [' offspring', ' will', ' live', ' longer', '\n', 'Answer', ':', ' C']
  Token at label pos 1497: id=340  decoded='C'
  OK: answer token 'C' matches expected 'C'

--- Sample 2  (expected answer: A) ---
  Sequence length (with padding): 1524
  Real tokens (non-pad):        

## 4. Training — Development Run

Train on `train_df`, evaluate on `val_df`. Use this to confirm the model learns and to pick the best epoch checkpoint. The `load_best_model_at_end=True` flag restores the lowest-val-loss checkpoint automatically.

In [9]:
# ── 5. Dev training run ───────────────────────────────────────────────────────

training_args_dev = TrainingArguments(
    output_dir="./qlora-dev",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    save_strategy="no",
    eval_strategy="no",
    logging_steps=25,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
)

model.config.use_cache = False   # silence gradient-checkpointing incompatibility warning

trainer_dev = Trainer(
    model=model,
    args=training_args_dev,
    train_dataset=train_ds,
    data_collator=_collate,
)

trainer_dev.train()
print("Dev training complete.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

Step,Training Loss
25,1010.262600
50,0.000000
75,0.000000
100,0.000000
125,0.000000
150,0.000000
175,0.000000
200,0.000000
225,0.000000
250,0.000000


Dev training complete.


In [10]:
# ── 5b. Save dev model (run immediately after training to avoid losing progress)
# Saves only the LoRA adapter weights (~20 MB), not the frozen base.
# Reload later with: model = PeftModel.from_pretrained(base, "./lora-dev-seed42")

model.save_pretrained("./lora-dev-seed42")
processor.save_pretrained("./lora-dev-seed42")
print("Dev adapter saved to ./lora-dev-seed42")

Dev adapter saved to ./lora-dev-seed42


## 5. Log-Likelihood Inference

**Why not `model.generate()`?**  
The baseline generates free text and tries to parse an answer letter from it. With a 500M model this is unreliable — the model may repeat the prompt, output multiple letters, or not output a letter at all.

**Log-likelihood scoring** computes `log P(letter | full_context)` for each candidate letter (A, B, C…) directly from the model's logits in a single forward pass, then picks the argmax. This is:
- Deterministic and always returns a valid answer
- Faster (no autoregressive decoding)
- More accurate — we compare the model's actual belief for each option

Since all `n` choice prompts share the same prefix and differ only in their last token, they can be batched together with no wasted compute.

In [11]:
# ── 6. Log-likelihood scoring ─────────────────────────────────────────────────

@torch.inference_mode()
def score_choices(model, processor, image: Image.Image, row: dict) -> list:
    """Return log P(answer_token | context) for each answer choice."""
    choices     = row["choices"]
    n           = len(choices)
    prompt_base = build_prompt(row, include_answer=False,
                               include_solution=TEST_HAS_SOLUTION)

    # Build one text per choice: they all end with " A", " B", " C" …
    full_texts = [prompt_base + f" {CHOICE_LETTERS[i]}" for i in range(n)]

    inputs = processor(
        text=full_texts,
        images=[image] * n,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
              for k, v in inputs.items()}

    eos_id = processor.tokenizer.eos_token_id
    logits = model(**inputs).logits  # (n, seq_len, vocab_size)

    log_probs = []
    for i in range(n):
        seq      = inputs["input_ids"][i].tolist()
        real_end = int(inputs["attention_mask"][i].sum().item())

        # Locate the answer letter: last real token that is not EOS
        answer_pos = real_end - 1
        while answer_pos >= 0 and seq[answer_pos] == eos_id:
            answer_pos -= 1

        # logit[t] predicts token[t+1], so the logit for answer_pos is at answer_pos-1
        pred_pos        = answer_pos - 1
        answer_token_id = seq[answer_pos]   # actual token ID from the sequence

        lp = torch.log_softmax(logits[i, pred_pos], dim=-1)[answer_token_id].item()
        log_probs.append(lp)

    return log_probs


def predict_one(model, processor, image: Image.Image, row: dict) -> int:
    lps = score_choices(model, processor, image, row)
    return int(torch.tensor(lps).argmax())


def run_inference(model, processor, df: pd.DataFrame,
                  data_dir: Path, img_size: int = IMG_SIZE) -> tuple:
    """Predict all rows in df. Returns (list_of_preds, accuracy_or_None)."""
    model.eval()
    preds, correct, total = [], 0, 0

    for _, row in df.iterrows():
        img  = Image.open(data_dir / "images" / row["image_path"]).convert("RGB")
        img  = img.resize((img_size, img_size), Image.BICUBIC)
        pred = predict_one(model, processor, img, row.to_dict())
        preds.append(pred)

        gt = row.get("answer", -1)
        if pd.notna(gt) and int(gt) >= 0:
            correct += (pred == int(gt))
            total   += 1

    acc = correct / total if total > 0 else None
    if acc is not None:
        print(f"Accuracy: {correct}/{total} = {acc:.4f}")
    return preds, acc


# ── Evaluate on validation set after dev training ─────────────────────────────
val_preds, val_acc = run_inference(model, processor, val_df, DATA_DIR)

Accuracy: 347/1048 = 0.3311


## 6. Final Training on train + val

With only ~4,157 labeled examples total, every sample matters. We reload a fresh base model and retrain on the full labeled set (train + val) for the same number of epochs as the best dev run. No early stopping — we use `NUM_EPOCHS` directly.

In [12]:
# ── 7. Final training (train + val) ──────────────────────────────────────────

del model
gc.collect()
torch.cuda.empty_cache()

model, processor = load_model_and_processor()
_collate = partial(collate_fn, proc=processor)   # rebind to the freshly loaded processor

full_df       = pd.concat([train_df, val_df], ignore_index=True)
full_train_ds = ScienceQADataset(
    full_df, DATA_DIR,
    is_train=True,
    augment_choices=True,
    include_solution=True,
)

training_args_final = TrainingArguments(
    output_dir="./qlora-final",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    save_strategy="epoch",
    eval_strategy="no",
    logging_steps=25,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
)

model.config.use_cache = False   # silence gradient-checkpointing incompatibility warning

trainer_final = Trainer(
    model=model,
    args=training_args_final,
    train_dataset=full_train_ds,
    data_collator=_collate,
)
trainer_final.train()

model.save_pretrained("./lora-final-seed42")
processor.save_pretrained("./lora-final-seed42")
print("Adapter saved to ./lora-final-seed42")

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Trainable params: 4,161,536 / 511,643,840 (0.813%)


Step,Training Loss
25,6095.984400
50,15308.862500
75,18337.010000


## 7. Generate Submission

In [ ]:
# ── 8. Predict test set and write submission.csv ──────────────────────────────

test_preds, _ = run_inference(model, processor, test_df, DATA_DIR)

submission = pd.DataFrame({
    "id":     test_df["id"],
    "answer": test_preds,
})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written — {len(submission)} rows")
print(submission.head(10))

## 8. Ensemble (3 Seeds) — Optional but Recommended

Train three independent LoRA adapters with different random seeds on the full dataset, then **average their log-probabilities** (not votes) before taking the argmax. Averaging log-probs is better than majority voting because it preserves each model's confidence.

Expected workflow:
1. Run **Section 6 + 7** with `SEED=42`  → save adapter to `lora-final-seed42`
2. Change `SEED=123` at the top, re-run **Section 6 + 7** → save to `lora-final-seed123`
3. Change `SEED=777`, repeat → save to `lora-final-seed777`
4. Run the ensemble cell below to produce the final `submission.csv`

In [ ]:
# ── 9. Ensemble inference ─────────────────────────────────────────────────────
#
# Load each saved adapter, collect log-probs for every test sample,
# average across adapters, pick argmax.
# Adjust the adapter_dirs list to match where you saved each run.

from peft import PeftModel

ADAPTER_DIRS = [
    "./lora-final-seed42",
    "./lora-final-seed123",
    "./lora-final-seed777",
]

bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16 — T4 emulates bf16, use native fp16
    bnb_4bit_use_double_quant=True,
)


@torch.inference_mode()
def ensemble_predict(adapter_dirs: list, df: pd.DataFrame,
                     data_dir: Path, img_size: int = IMG_SIZE) -> list:
    # Accumulate log-prob tensors: shape (len(df), max_choices)
    all_log_probs = None  # will be list of lists, one per row

    for adapter_dir in adapter_dirs:
        print(f"\nLoading adapter: {adapter_dir}")
        proc = AutoProcessor.from_pretrained(adapter_dir)
        if proc.tokenizer.pad_token is None:
            proc.tokenizer.pad_token = proc.tokenizer.eos_token

        base = AutoModelForVision2Seq.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config_inf,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
        mdl = PeftModel.from_pretrained(base, adapter_dir)
        mdl.eval()

        model_log_probs = []
        for _, row in df.iterrows():
            img = Image.open(data_dir / "images" / row["image_path"]).convert("RGB")
            img = img.resize((img_size, img_size), Image.BICUBIC)
            lps = score_choices(mdl, proc, img, row.to_dict())
            model_log_probs.append(lps)

        if all_log_probs is None:
            all_log_probs = [[lp] for lp in model_log_probs]
        else:
            for i, lp in enumerate(model_log_probs):
                all_log_probs[i].append(lp)

        # Free VRAM between adapters
        del mdl, base
        gc.collect()
        torch.cuda.empty_cache()

    # Average log-probs across adapters and argmax
    preds = []
    for row_lps in all_log_probs:
        stacked = torch.tensor(row_lps)          # (num_adapters, num_choices)
        avg     = stacked.mean(dim=0)
        preds.append(int(avg.argmax()))

    return preds


# Only run this cell after you have trained all three adapters.
# Comment out or skip if running a single-seed submission.

ensemble_preds = ensemble_predict(ADAPTER_DIRS, test_df, DATA_DIR)

submission_ensemble = pd.DataFrame({
    "id":     test_df["id"],
    "answer": ensemble_preds,
})
submission_ensemble.to_csv("submission.csv", index=False)
print(f"Ensemble submission written — {len(submission_ensemble)} rows")
print(submission_ensemble.head(10))